# Playground Series S6E9 — Feature Engineering

Builds on `01_eda.ipynb` findings: `Environmental_Concern_Level` and `Annual_Income_USD`
are the strongest linear correlates; categoricals (`Home_Charging_Possible`, `City_Type`,
`Current_Car_Type`) show purchase-rate separation. Encode categoricals, derive a few
domain features, and write a modeling-ready parquet.


In [1]:
import pandas as pd
import numpy as np

train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

train["Will_Buy_EV_bin"] = (train["Will_Buy_EV"] == "Yes").astype(int)
target = "Will_Buy_EV_bin"

train.shape, test.shape


((668665, 16), (286571, 14))

## Binary yes/no columns → 0/1

In [2]:
binary_cols = ["Home_Charging_Possible", "Subsidy_Available"]
for df in (train, test):
    for col in binary_cols:
        df[col] = (df[col] == "Yes").astype(int)

train[binary_cols].head()


,Home_Charging_Possible,Subsidy_Available
0,1,0
1,1,0
2,0,1
3,1,0
4,1,0


## Ordinal encode `Range_Anxiety_Level`
Check the category order first — treat as ordinal if it reads Low/Medium/High.

In [3]:
print(train["Range_Anxiety_Level"].unique())
anxiety_order = {"Low": 0, "Medium": 1, "High": 2}
for df in (train, test):
    df["Range_Anxiety_Level_ord"] = df["Range_Anxiety_Level"].map(anxiety_order)

train[["Range_Anxiety_Level", "Range_Anxiety_Level_ord"]].drop_duplicates()


<ArrowStringArray>
['Low', 'Medium', 'High']
Length: 3, dtype: str


,Range_Anxiety_Level,Range_Anxiety_Level_ord
0,Low,0
7,Medium,1
219,High,2


## One-hot encode remaining nominal categoricals

In [4]:
nominal_cols = ["Gender", "City_Type", "Current_Car_Type"]
train_enc = pd.get_dummies(train, columns=nominal_cols, drop_first=True)
test_enc = pd.get_dummies(test, columns=nominal_cols, drop_first=True)

# align columns (test may be missing a dummy level present in train, or vice versa)
train_enc, test_enc = train_enc.align(test_enc, join="left", axis=1, fill_value=0)
test_enc = test_enc.drop(columns=[c for c in ["Will_Buy_EV", "Will_Buy_EV_bin"] if c in test_enc.columns])

train_enc.shape, test_enc.shape


((668665, 21), (286571, 19))

## Derived features

In [5]:
for df in (train_enc, test_enc):
    df["Charging_Access_Score"] = (
        df["Charging_Stations_Near_Home"] + df["Charging_Stations_Near_Work"]
        + df["Home_Charging_Possible"]
    )
    df["Commute_to_Income"] = df["Daily_Commute_km"] / (df["Annual_Income_USD"] / 1000)
    df["Income_per_Car"] = df["Annual_Income_USD"] / df["Number_of_Cars_Owned"].replace(0, 1)

train_enc[["Charging_Access_Score", "Commute_to_Income", "Income_per_Car"]].describe()


,Charging_Access_Score,Commute_to_Income,Income_per_Car
count,668665.000000,668665.000000,668665.000000
mean,12.828663,0.444495,58754.583977
std,7.750677,0.363147,31488.101505
min,0.000000,0.026621,7500.000000
25%,6.000000,0.179737,33897.000000
50%,12.000000,0.389208,50559.500000
75%,18.000000,0.589092,81082.000000
max,34.000000,3.273333,188549.000000


## Save modeling-ready data

In [6]:
feature_cols = [c for c in train_enc.columns if c not in ("id", "Will_Buy_EV", "Will_Buy_EV_bin", "Range_Anxiety_Level")]
train_enc[feature_cols + ["Will_Buy_EV_bin"]].to_parquet("../data/train_features.parquet", index=False)
test_enc[[c for c in feature_cols if c != "Will_Buy_EV_bin"]].assign(id=test["id"]).to_parquet("../data/test_features.parquet", index=False)
print(f"{len(feature_cols)} features saved")
feature_cols


20 features saved


['Age',
 'Annual_Income_USD',
 'Daily_Commute_km',
 'Number_of_Cars_Owned',
 'Charging_Stations_Near_Home',
 'Charging_Stations_Near_Work',
 'Environmental_Concern_Level',
 'Home_Charging_Possible',
 'Subsidy_Available',
 'Range_Anxiety_Level_ord',
 'Gender_Male',
 'Gender_Other',
 'City_Type_Suburban',
 'City_Type_Urban',
 'Current_Car_Type_SUV',
 'Current_Car_Type_Sedan',
 'Current_Car_Type_Truck',
 'Charging_Access_Score',
 'Commute_to_Income',
 'Income_per_Car']

## Key Takeaways

- `Range_Anxiety_Level` treated as ordinal (Low<Medium<High); confirm this order matches
  domain intuition before trusting the ordinal encoding in modeling.
- One-hot encoding + train/test column alignment prevents a shape mismatch at inference time.
- `Charging_Access_Score` and `Commute_to_Income` are hypotheses to validate via feature
  importance in `03_modeling.ipynb` — drop if they add no signal over the raw columns.
